# CatBoost: прогноз повторной госпитализации

Цель - сравнить CatBoost с baseline-моделью Logistic Regression на одинаковых признаках и одинаковых групповых фолдах.

Основная метрика - Average Precision (PR-AUC).
Дополнительная метрика - ROC-AUC.

Все записи одного пациента должны находиться внутри одного фолда.

In [12]:
import joblib
from matplotlib import pyplot as plt
import json
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, accuracy_score, \
    average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix
)



In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.metrics import (
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)

from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_validate,
)

In [9]:
candidates = [Path.cwd(), *Path.cwd().parents]

project_root = next(
    path
    for path in candidates
    if (
        path
        / "data"
        / "interim"
        / "modeling_dataset.parquet"
    ).exists()
)

data_path = (
    project_root
    / "data"
    / "interim"
    / "modeling_dataset.parquet"
)

data = pd.read_parquet(data_path)

train_data = (
    data.loc[data["split"].eq("train")]
    .drop(columns="split")
    .copy()
)

validation_data = (
    data.loc[data["split"].eq("validation")]
    .drop(columns="split")
    .copy()
)

test_data = (
    data.loc[data["split"].eq("test")]
    .drop(columns="split")
    .copy()
)

print(f"Train:      {train_data.shape}")
print(f"Validation: {validation_data.shape}")
print(f"Test:       {test_data.shape}")

Train:      (69519, 50)
Validation: (14911, 50)
Test:       (14913, 50)


In [10]:
target_column = "target_30d"

excluded_features = [
    "target_30d",
    "readmitted",
    "encounter_id",
    "patient_nbr",
    "examide",
    "citoglipton",
]

groups_train = train_data["patient_nbr"].copy() #сохраняем пациентов для групповой кросс-валидации


X_train = train_data.drop(columns=excluded_features)
y_train = train_data[target_column].copy()

X_validation = validation_data.drop(columns=excluded_features)
y_validation = validation_data[target_column].copy()

X_test = test_data.drop(columns=excluded_features)
y_test = test_data[target_column].copy()

In [14]:
coded_categorical_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

for frame in [X_train, X_validation, X_test]:
    frame[coded_categorical_features] = (
        frame[coded_categorical_features].astype("string")
    )

    numeric_features = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)
coded_categorical_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

categorical_features = (
    X_train.select_dtypes(exclude="number").columns.tolist()
    + coded_categorical_features
)
categorical_features = list(dict.fromkeys(categorical_features))

numeric_features = [
    column
    for column in X_train.columns
    if column not in categorical_features
]

In [15]:
def prepare_for_catboost(data):
    prepared_data = data.copy()

    for column in categorical_features:
        prepared_data[column] = (
            prepared_data[column]
            .astype("object")
            .fillna("Unknown")
            .astype(str)
        )

    return prepared_data


X_train_cb = prepare_for_catboost(X_train)
X_validation_cb = prepare_for_catboost(X_validation)
X_test_cb = prepare_for_catboost(X_test)

In [21]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    thread_count=1,
)

In [22]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

catboost_cv_results = cross_validate(
    catboost_model,
    X_train_cb,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
       params={
        "cat_features": categorical_features,
    },
    n_jobs=-1,
)

In [24]:
metrics = [
    "roc_auc",
    "pr_auc",
    "recall",
    "precision",
    "f1",
]

catboost_cv_summary = pd.DataFrame(
    {
        "mean": [
            catboost_cv_results[f"test_{metric}"].mean()
            for metric in metrics
        ],
        "std": [
            catboost_cv_results[f"test_{metric}"].std()
            for metric in metrics
        ],
    },
    index=metrics,
)

catboost_cv_summary

,mean,std
roc_auc,0.677157,0.005982
pr_auc,0.238544,0.008059
recall,0.021221,0.004175
precision,0.602360,0.076695
f1,0.040926,0.007782


### Вывод

CatBoost превосходит логистическую регрессию по ROC-AUC (0.677 против
0.649) и PR-AUC (0.239 против 0.208). Небольшое стандартное отклонение
показывает устойчивость результата между фолдами.

При стандартном пороге 0.5 модель обладает высокой точностью
положительных прогнозов (precision = 0.602), но обнаруживает только
около 2% повторных госпитализаций (recall = 0.021). Поэтому порог 0.5
не подходит для поставленной задачи.

CatBoost выбирается как основная модель

In [25]:
selected_catboost = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

selected_catboost.fit(
    X_train_cb,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_validation_cb, y_validation),
    early_stopping_rounds=100,
    use_best_model=True,
)

validation_probability = selected_catboost.predict_proba(
    X_validation_cb
)[:, 1]

print("Лучшее число деревьев:", selected_catboost.get_best_iteration())

0:	test: 0.4996583	best: 0.4996583 (0)	total: 160ms	remaining: 2m 39s
100:	test: 0.6737000	best: 0.6737000 (100)	total: 13.5s	remaining: 1m 59s
200:	test: 0.6788526	best: 0.6788526 (200)	total: 21.6s	remaining: 1m 25s
300:	test: 0.6808668	best: 0.6808684 (295)	total: 33.7s	remaining: 1m 18s
400:	test: 0.6825670	best: 0.6825689 (397)	total: 48.6s	remaining: 1m 12s
500:	test: 0.6836695	best: 0.6836695 (500)	total: 58.6s	remaining: 58.4s
600:	test: 0.6842677	best: 0.6842677 (600)	total: 1m 14s	remaining: 49.2s
700:	test: 0.6849920	best: 0.6849923 (699)	total: 1m 25s	remaining: 36.6s
800:	test: 0.6852774	best: 0.6852780 (799)	total: 1m 40s	remaining: 25.1s
900:	test: 0.6854268	best: 0.6854314 (895)	total: 1m 51s	remaining: 12.2s
999:	test: 0.6855241	best: 0.6856789 (974)	total: 2m 2s	remaining: 0us

bestTest = 0.6856789167
bestIteration = 974

Shrink model to first 975 iterations.
Лучшее число деревьев: 974


In [26]:
precision, recall, thresholds = precision_recall_curve(
    y_validation,
    validation_probability,
)

threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
})

threshold_results["f1"] = (
    2
    * threshold_results["precision"]
    * threshold_results["recall"]
    / (
        threshold_results["precision"]
        + threshold_results["recall"]
        + 1e-10
    )
)

best_threshold_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold_row

threshold    0.153694
precision    0.243629
recall       0.412069
f1           0.306214
Name: 11968, dtype: float64

In [27]:
best_threshold = best_threshold_row["threshold"]

validation_prediction = (
    validation_probability >= best_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    validation_prediction,
).ravel()

validation_metrics = pd.Series({
    "threshold": best_threshold,
    "roc_auc": roc_auc_score(
        y_validation,
        validation_probability,
    ),
    "pr_auc": average_precision_score(
        y_validation,
        validation_probability,
    ),
    "precision": tp / (tp + fp),
    "recall": tp / (tp + fn),
    "specificity": tn / (tn + fp),
    "f1": 2 * tp / (2 * tp + fp + fn),
    "npv": tn / (tn + fn),
})

display(validation_metrics.round(3))

print(
    classification_report(
        y_validation,
        validation_prediction,
        digits=3,
        zero_division=0,
    )
)

print("Confusion matrix:")
display(
    pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Факт: нет", "Факт: да"],
        columns=["Прогноз: нет", "Прогноз: да"],
    )
)

threshold      0.154
roc_auc        0.686
pr_auc         0.259
precision      0.244
recall         0.412
specificity    0.831
f1             0.306
npv            0.915
dtype: float64

              precision    recall  f1-score   support

           0      0.915     0.831     0.871     13171
           1      0.244     0.412     0.306      1740

    accuracy                          0.782     14911
   macro avg      0.579     0.622     0.588     14911
weighted avg      0.836     0.782     0.805     14911

Confusion matrix:


,Прогноз: нет,Прогноз: да
Факт: нет,10945,2226
Факт: да,1023,717


При пороге 0.154 CatBoost обнаруживает 41.2% повторных госпитализаций
с precision 24.4% и F1 0.306.

По сравнению с логистической регрессией CatBoost значительно сокращает
число ложноположительных прогнозов: 2 226 против 2 738. Количество найденных повторных госпитализаций уменьшается только
с 741 до 717.

CatBoost показывает лучший баланс precision и recall. Однако для скринингового медицинского применения
recall 41.2% может быть недостаточным, поэтому дополнительно необходимо
рассмотреть более низкие пороги.

In [28]:
target_recalls = [0.40, 0.50, 0.60, 0.70]
threshold_scenarios = []

for target_recall in target_recalls:
    candidates = threshold_results[
        threshold_results["recall"] >= target_recall
    ]

    best_candidate = candidates.loc[
        candidates["precision"].idxmax()
    ].copy()

    best_candidate["target_recall"] = target_recall
    threshold_scenarios.append(best_candidate)

threshold_scenarios = pd.DataFrame(threshold_scenarios)

threshold_scenarios[
    ["target_recall", "threshold", "precision", "recall", "f1"]
].round(3)

,target_recall,threshold,precision,recall,f1
12075,0.4,0.156,0.245,0.400,0.304
10871,0.5,0.132,0.215,0.500,0.301
9397,0.6,0.112,0.190,0.601,0.288
7857,0.7,0.097,0.173,0.701,0.277


Зафиксированы два сценария применения модели:

1. сбалансированный порог, максимизирующий F1
2. скрининговый порог, обеспечивающий recall не менее 60%.

Скрининговый сценарий снижает precision до 19%, но дает обнаружить
около 60% повторных госпитализаций. Финальный порог можем определить, если узнаем стоимость пропущенных клинических случаев.

In [31]:
test_probability = selected_catboost.predict_proba(
    X_test_cb
)[:, 1]


def evaluate_test_scenario(name, threshold):
    prediction = (
        test_probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        prediction,
    ).ravel()

    return {
        "scenario": name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(
            y_test,
            test_probability,
        ),
        "pr_auc": average_precision_score(
            y_test,
            test_probability,
        ),
        "precision": precision_score(
            y_test,
            prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_test,
            prediction,
        ),
        "specificity": tn / (tn + fp),
        "f1": f1_score(
            y_test,
            prediction,
        ),
        "accuracy": accuracy_score(
            y_test,
            prediction,
        ),
        "alert_rate": prediction.mean(),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


test_results = pd.DataFrame([
    evaluate_test_scenario(name, threshold)
    for name, threshold in selected_thresholds.items()
]).set_index("scenario")

test_results

,threshold,roc_auc,pr_auc,precision,recall,specificity,f1,accuracy,alert_rate,tn,fp,fn,tp
scenario,,,,,,,,,,,,,
balanced_f1,0.153694,0.683376,0.239173,0.227510,0.395054,0.832214,0.288737,0.783612,0.193053,11031,2224,1003,655
screening_recall_60,0.112162,0.683376,0.239173,0.179285,0.589867,0.662241,0.274989,0.654194,0.365788,8778,4477,680,978


При пороге 0.154 модель находит около 40% пациентов, которые повторно
попадут в больницу. При этом дополнительная проверка потребуется примерно
для 19% всех пациентов.
Если снизить порог до 0.112, модель сможет найти уже 59% повторных
госпитализаций. Однако в таком случае проверять придётся около 37%
пациентов, а количество ложных предупреждений заметно возрастёт.
Таким образом, более низкий порог помогает пропустить меньше пациентов
из группы риска, но увеличивает нагрузку на медицинский персонал.
